# Agent Demo — Tool-Calling Q&A on Vertex AI

Demonstrates a LangGraph ReAct agent that decides *how* to answer each question:
by searching the GCP documentation corpus, or by answering from Gemini's training
knowledge directly.

**The two tools:**

| Tool | When used | Mechanism |
|---|---|---|
| `search_docs` | GCP/Vertex AI specific questions | Full RAG pipeline — retrieval + generation from corpus |
| `answer_direct` | General ML/programming questions | Gemini with no context injection |

**The Bedrock parallel:** This is the same pattern as Bedrock Agents with a retrieval
tool — the agent decides whether to invoke the knowledge base or answer directly. LangGraph
makes the routing logic explicit and inspectable: you can see exactly which tool was called,
with what arguments, and why.

**Sections:**
1. Setup & Auth
2. Tool Definitions & Graph Construction
3. Demo Queries — docs tool wins
4. Demo Queries — direct tool wins
5. Analysis

## Section 1 — Setup & Auth

In [ ]:
import os
import sys
import warnings
from pathlib import Path

from dotenv import load_dotenv

# ── Backend toggle ─────────────────────────────────────────────────────────────
#   "gemini_api" — free Gemini API (GEMINI_API_KEY in .env)
#   "vertex_ai"  — Vertex AI via ADC (GCP_PROJECT_ID in .env)
BACKEND = "vertex_ai"
# ───────────────────────────────────────────────────────────────────────────────

load_dotenv(dotenv_path="../.env")
sys.path.insert(0, str(Path("../src").resolve()))
warnings.filterwarnings("ignore")

REPO_ROOT  = Path.cwd().parent
CORPUS_DIR = REPO_ROOT / "corpus"

from google import genai
from embedder import load_embeddings

if BACKEND == "vertex_ai":
    project  = os.environ.get("GCP_PROJECT_ID")
    location = os.environ.get("GCP_LOCATION", "us-central1")
    assert project, "GCP_PROJECT_ID not set — check .env"
    client = genai.Client(vertexai=True, project=project, location=location)
    print(f"Backend  : Vertex AI")
    print(f"Project  : {project}  Location: {location}")
else:
    api_key = os.environ.get("GEMINI_API_KEY")
    assert api_key, "GEMINI_API_KEY not set — check .env"
    client = genai.Client(api_key=api_key)
    print(f"Backend  : Gemini API (free tier)")

print("\nLoading corpus embeddings...")
embeddings, chunks = load_embeddings(CORPUS_DIR)
print(f"  {embeddings.shape[0]} chunks, {embeddings.shape[1]} dims")

## Section 2 — Tool Definitions & Graph Construction

The agent is a standard LangGraph ReAct loop:

```
user query → agent (LLM) → tool call → tool result → agent → final answer
```

Gemini sees the tool schemas and decides which to invoke. The loop continues
until Gemini produces a final answer with no pending tool calls.

In [ ]:
from agent import build_agent, run_query

rag_agent = build_agent(client, embeddings, chunks, backend=BACKEND)
print("Agent ready.")
print()
print("Tools available:")
for tool in rag_agent.tools:
    print(f"  {tool.name:<20} {tool.description[:80]}")

## Section 3 — Demo: `search_docs` wins

Questions that require specific Vertex AI documentation — pricing tables, API limits,
SDK behavior. Gemini's training data may be outdated or imprecise; the corpus has the
authoritative answer.

In [ ]:
def show_result(result: dict) -> None:
    sep = "─" * 72
    tools_str = " → ".join(result["tools_used"]) if result["tools_used"] else "none"
    print(sep)
    print(f"  QUESTION   : {result['question']}")
    print(f"  TOOL USED  : {tools_str}")
    print(sep)
    print()
    for line in result["answer"].strip().splitlines():
        print(f"  {line}")
    print()


queries_docs = [
    "What are the six steps in the Vertex AI RAG Engine pipeline?",
    "What is the output dimensionality of gemini-embedding-001?",
    "What are the per-request limits for the Vertex AI Text Embeddings API?",
]

print("Running search_docs demo queries...\n")
docs_results = []
for q in queries_docs:
    r = run_query(rag_agent, q)
    docs_results.append(r)
    show_result(r)

## Section 4 — Demo: `answer_direct` wins

Questions where the corpus adds no value — general ML concepts, Python fundamentals,
or topics outside the GCP documentation set. The agent should recognize these and
answer from training knowledge without touching the corpus.

In [ ]:
queries_direct = [
    "What is the difference between precision and recall in machine learning?",
    "What does the Python `@` operator do when used between two numpy arrays?",
    "Explain the transformer attention mechanism in two sentences.",
]

print("Running answer_direct demo queries...\n")
direct_results = []
for q in queries_direct:
    r = run_query(rag_agent, q)
    direct_results.append(r)
    show_result(r)

## Section 5 — Analysis

Summarise routing decisions across all queries and assess whether the agent chose
the right tool in each case.

In [ ]:
all_results = docs_results + direct_results
expected = (
    ["search_docs"] * len(queries_docs) +
    ["answer_direct"] * len(queries_direct)
)

print("  ROUTING ANALYSIS")
print("  " + "─" * 68)
print(f"  {'Question':<52} {'Expected':<15} {'Got':<15} {'✓'}")
print("  " + "─" * 68)

correct = 0
for r, exp in zip(all_results, expected):
    got   = r["tool_used"]
    match = got == exp
    if match:
        correct += 1
    mark  = "✓" if match else "✗"
    q_short = r["question"][:50] + "..." if len(r["question"]) > 50 else r["question"]
    print(f"  {q_short:<52} {exp:<15} {got:<15} {mark}")

print("  " + "─" * 68)
print(f"  Correct: {correct}/{len(all_results)}")
print()
print("  The Bedrock parallel: Bedrock Agents with a retrieval tool works the same way.")
print("  LangGraph makes the routing decision explicit — each tool call is logged,")
print("  inspectable, and auditable. In production you'd add tracing (LangSmith or")
print("  Cloud Trace) to monitor routing quality over time.")